<a href="https://colab.research.google.com/github/swastiksureshcs24-sudo/Ai-lab/blob/main/1BM24CS304_WEEK7_FOL_IMPLEMENTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
def is_variable(term):
    """Variables are lowercase strings (e.g., 'x', 'y')."""
    return isinstance(term, str) and len(term) > 0 and term.islower()

def is_constant(term):
    """Constants are uppercase/capitalized strings (e.g., 'John', 'A')."""
    return isinstance(term, str) and len(term) > 0 and not term.islower()

def occurs_in(var, expr):
    """Checks if variable 'var' occurs anywhere inside 'expr'."""
    if var == expr:
        return True
    if isinstance(expr, list):
        return any(occurs_in(var, sub_expr) for sub_expr in expr)
    return False

def apply_substitution(subst, expr):
    """Applies a list of substitutions [(val, var), ...] to an expression."""
    if not subst:
        return expr
    for val, var in subst:
        expr = substitute_single(val, var, expr)
    return expr

def substitute_single(val, var, expr):
    """Replaces all occurrences of 'var' with 'val' in 'expr'."""
    if isinstance(expr, str):
        return val if expr == var else expr
    if isinstance(expr, list):
        return [substitute_single(val, var, item) for item in expr]
    return expr

def unify(psi1, psi2):
    """Unify(Ψ1, Ψ2) returning a list of (replacement, variable) tuples or 'FAILURE'."""
    if isinstance(psi1, str) or isinstance(psi2, str):
        if psi1 == psi2:
            return []
        elif is_variable(psi1):
            return "FAILURE" if occurs_in(psi1, psi2) else [(psi2, psi1)]
        elif is_variable(psi2):
            return "FAILURE" if occurs_in(psi2, psi1) else [(psi1, psi2)]
        else:
            return "FAILURE"

    if psi1[0] != psi2[0] or len(psi1) != len(psi2):
        return "FAILURE"

    subst = []
    l1 = list(psi1)
    l2 = list(psi2)

    for i in range(1, len(l1)):
        s = unify(l1[i], l2[i])
        if s == "FAILURE":
            return "FAILURE"
        if s != []:
            for j in range(i + 1, len(l1)):
                l1[j] = apply_substitution(s, l1[j])
                l2[j] = apply_substitution(s, l2[j])
            subst = [(apply_substitution(s, val), var) for val, var in subst]
            subst.extend(s)

    return subst

def print_xy_only(iteration_num, cond1, cond2, result):
    """Prints the two input conditions, status, and values for x and y."""
    print(f"Iteration {iteration_num}:")
    print(f"  Condition eats: {cond1}")
    print(f"  Condition eats: {cond2}")

    if result == "FAILURE":
        print("  Status: FAILURE (No valid x and y values)\n")
        return

    print("  Status: SUCCESS")
    xy_bindings = {var: val for val, var in result if var in ('x', 'y')}

    print(f"  x = {xy_bindings.get('x', 'Unbound')}")
    print(f"  y = {xy_bindings.get('y', 'Unbound')}\n")

if __name__ == "__main__":
    # Iteration 1
    c1_1 = ['P', 'x', ['f', 'y']]
    c1_2 = ['P', 'John', ['f', 'z']]
    print_xy_only(1, c1_1, c1_2, unify(c1_1, c1_2))

    # Iteration 2
    c2_1 = ['P', 'x', 'x']
    c2_2 = ['P', 'y', ['f', 'y']]
    print_xy_only(2, c2_1, c2_2, unify(c2_1, c2_2))

    # Iteration 3
    c3_1 = ['Knows', 'x', ['Father', 'x'], 'y']
    c3_2 = ['Knows', ['Mother', 'z'], ['Father', ['Mother', 'Alice']], 'Bob']
    print_xy_only(3, c3_1, c3_2, unify(c3_1, c3_2))

Iteration 1:
  Condition eats: ['P', 'x', ['f', 'y']]
  Condition eats: ['P', 'John', ['f', 'z']]
  Status: SUCCESS
  x = John
  y = z

Iteration 2:
  Condition eats: ['P', 'x', 'x']
  Condition eats: ['P', 'y', ['f', 'y']]
  Status: FAILURE (No valid x and y values)

Iteration 3:
  Condition eats: ['Knows', 'x', ['Father', 'x'], 'y']
  Condition eats: ['Knows', ['Mother', 'z'], ['Father', ['Mother', 'Alice']], 'Bob']
  Status: SUCCESS
  x = ['Mother', 'Alice']
  y = Bob

